# Engineering production-readiness gates

This example configures the process-to-engineering loop from an explicit project policy, compiles the coordinated DEXPI package, and inspects the evidence gates. It deliberately does not fabricate independent benchmarks, HAZOP/LOPA decisions, CAE qualification, pilot acceptance, or release approval, so the package remains unqualified for production use.

In [ ]:
import json, os, shutil, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=not (ROOT / 'target/classes').exists(), verbose=False))
import jpype
import pandas as pd
from IPython.display import display
JClass, JProxy = ns.JClass, jpype.JProxy
print('Loaded production-readiness API from', ROOT)

In [ ]:
SystemSrkEos = JClass('neqsim.thermo.system.SystemSrkEos')
Stream = JClass('neqsim.process.equipment.stream.Stream')
Separator = JClass('neqsim.process.equipment.separator.Separator')
Compressor = JClass('neqsim.process.equipment.compressor.Compressor')
AdiabaticPipe = JClass('neqsim.process.equipment.pipeline.AdiabaticPipe')
ProcessSystem = JClass('neqsim.process.processmodel.ProcessSystem')
fluid = SystemSrkEos(300.0, 50.0)
for component, amount in [('methane', 0.92), ('ethane', 0.04), ('n-heptane', 0.04)]:
    fluid.addComponent(component, amount)
fluid.setMixingRule('classic')
feed = Stream('FEED', fluid); feed.setFlowRate(8000.0, 'kg/hr')
separator = Separator('INLET-SEP', feed)
compressor = Compressor('EXPORT-COMP', separator.getGasOutStream())
compressor.setOutletPressure(80.0, 'bara'); compressor.setPolytropicEfficiency(0.78)
export_line = AdiabaticPipe('EXPORT-LINE', compressor.getOutletStream())
export_line.setLength(1000.0); export_line.setDiameter(0.2027); export_line.setPipeWallRoughness(4.6e-5)
process = ProcessSystem()
for unit in [feed, separator, compressor, export_line]: process.add(unit)
process.run()
print('Base process converged')

In [ ]:
NorsokBuilder = JClass('neqsim.process.engineering.NorsokOffshoreEngineeringBuilder')
DesignCase = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase')
Configurator = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase$Configurator')
AutoPolicy = JClass('neqsim.process.engineering.production.EngineeringAutoConfigurationPolicy')
Simulator = JClass('neqsim.process.engineering.ProcessToEngineeringSimulator')
project = NorsokBuilder.from_('Production-readiness example', process).projectId('production-readiness-notebook').build()
for case_id, flow, priority in [('turndown', 5000.0, 5), ('normal', 8000.0, 10), ('maximum', 12000.0, 20)]:
    def configure(case_process, flow=flow):
        case_process.getUnit('FEED').setFlowRate(flow, 'kg/hr')
    project.addDesignCase(DesignCase(case_id, case_id, DesignCase.Type.CUSTOM, JProxy(Configurator, dict={'configure': configure})).setPriority(priority))
drivers = jpype.JArray(jpype.JDouble)([500, 1000, 2000, 3000, 5000, 7500, 10000])
policy = AutoPolicy('offshore-gas-policy', 'A')
policy.addInletCompressionExportSlice('INLET-SEP', 'EXPORT-COMP', 'EXPORT-LINE', '', 'PIT-100', 800.0, 0.107, 120.0, 25.0, 5.0, 0.10, drivers)
result = Simulator.run(project, policy, 3)
configured = project.getProductionReadinessBasis().getAutoConfigurationResult()
assert configured.isComplete() and result.getEngineeringDesignLoopResult().isConverged()
print('Explicit configuration complete; converged iterations:', result.getEngineeringDesignLoopResult().getIterations().size())

In [ ]:
loop = result.getEngineeringDesignLoopResult()
final_iteration = loop.getIterations().get(loop.getIterations().size() - 1)
method_keys = sorted({str(module.getMethod()) + '@' + str(module.getMethodVersion()) for module in final_iteration.getModuleResults()})
pd.DataFrame({'executed method/version requiring evidence': method_keys})

In [ ]:
Benchmark = JClass('neqsim.process.engineering.production.EngineeringValidationBenchmark')
BenchmarkSuite = JClass('neqsim.process.engineering.production.EngineeringBenchmarkSuite')
SourceClass = JClass('neqsim.process.engineering.production.EngineeringValidationBenchmark$SourceClass')
first_method, first_version = method_keys[0].rsplit('@', 1)
regression = Benchmark.builder('internal-regression', first_method, first_version).source(SourceClass.REGRESSION_BASELINE, 'internal snapshot', 'A').check('example value', 1.0, 1.0, 'fraction', 0.0, 0.0).build()
regression_report = BenchmarkSuite('demonstration', 'A').requireMethod(method_keys[0]).add(regression).evaluate()
assert not regression_report.isPassed()
print('Regression-only evidence rejected for:', list(regression_report.getMissingQualifyingMethods()))

In [ ]:
Compiler = JClass('neqsim.process.engineering.deliverables.EngineeringDeliverableCompiler')
Paths = JClass('java.nio.file.Paths')
package_dir = Path('/tmp/neqsim-engineering-production-readiness-package')
shutil.rmtree(package_dir, ignore_errors=True)
compiled = Compiler.compile(project, Paths.get(str(package_dir)))
readiness = json.loads((package_dir / 'engineering-production-readiness.json').read_text())
gate_rows = [{'gate': name, **value} for name, value in readiness['gates'].items()]
display(pd.DataFrame(gate_rows))
print('Maturity:', readiness['maturityLevel'])
print('Preliminary production ready:', readiness['preliminaryProductionReady'])
print('Fitness for construction:', readiness['fitnessForConstruction'])
assert compiled.getValidationReport().isValid()
assert readiness['fitnessForConstruction'] is False
assert readiness['preliminaryProductionReady'] is False

A project can reach `QUALIFIED_FEED_SUPPORT` only after controlled evidence is attached for every failed gate. The simulator cannot generate its own HAZOP/LOPA conclusions, independent review, commercial-tool result, pilot acceptance, or accountable approval. Even a fully green preliminary assessment remains explicitly not fit for construction.